<a href="https://colab.research.google.com/github/Kaluvai1203/CSA6102---Lab-/blob/main/Untitled29.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from datetime import datetime, timedelta

def parse_time(t):
    return datetime.strptime(t, "%Y-%m-%d %H:%M:%S")


def detect_bruteforce(events, threshold=5, window_minutes=2):
    """
    Detect brute-force login attempts:
    >= threshold failed logons (4625)
    for the same account within window_minutes,
    and report whether a successful logon (4624) followed.
    """
    events = sorted(events, key=lambda e: parse_time(e["timestamp"]))
    by_account = {}
    for e in events:
        by_account.setdefault(e["account"], []).append(e)

    results = {}

    for account, acc_events in by_account.items():
        failures = [e for e in acc_events if e["event_id"] == 4625]
        successes = [e for e in acc_events if e["event_id"] == 4624]

        flagged = False
        for i in range(len(failures)):
            window_start = parse_time(failures[i]["timestamp"])
            window_end = window_start + timedelta(minutes=window_minutes)

            count = sum(
                1
                for f in failures
                if window_start <= parse_time(f["timestamp"]) <= window_end
            )

            if count >= threshold:
                flagged = True
                break
        success_after = False
        if flagged:
            for s in successes:
                if parse_time(s["timestamp"]) > window_start:
                    success_after = True
                    break

        results[account] = {
            "bruteforce_detected": flagged,
            "successful_login_after": success_after,
        }

    return results
events = [
    {"timestamp": "2026-08-04 10:00:00", "event_id": 4625, "account": "alice"},
    {"timestamp": "2026-08-04 10:00:20", "event_id": 4625, "account": "alice"},
    {"timestamp": "2026-08-04 10:00:40", "event_id": 4625, "account": "alice"},
    {"timestamp": "2026-08-04 10:01:00", "event_id": 4625, "account": "alice"},
    {"timestamp": "2026-08-04 10:01:30", "event_id": 4625, "account": "alice"},
    {"timestamp": "2026-08-04 10:02:00", "event_id": 4624, "account": "alice"},
    {"timestamp": "2026-08-04 10:10:00", "event_id": 4625, "account": "bob"},
]
result = detect_bruteforce(events)
for account, info in result.items():
    print(f"Account: {account}")
    print(f"Brute Force Detected: {info['bruteforce_detected']}")
    print(f"Successful Login After Attack: {info['successful_login_after']}")
    print("-" * 40)

Account: alice
Brute Force Detected: True
Successful Login After Attack: True
----------------------------------------
Account: bob
Brute Force Detected: False
Successful Login After Attack: False
----------------------------------------
